In [26]:
import geopandas as gpd
import pandas as pd
import mgrs

from ipyleaflet import Map, GeoData, basemaps, LayersControl
from shapely.geometry import Point

In [2]:
df = pd.read_excel('data/refpoints.xlsx')

In [63]:
def from_lon_lat(coordinate: str) -> Point:
    lon = coordinate.split(', ')[0]
    lat = coordinate.split(', ')[1]
    return Point(lon,lat)

def from_mgrs(coordinate: str) -> Point:
    m = mgrs.MGRS()
    lat, lon = m.toLatLon(coordinate)
    return Point(lon,lat)

def dms_to_dd(d, m, s):
    dd = d + float(m)/60 + float(s)/3600
    return dd

def from_dms(coordinate: str) -> Point:
    lat, lon = coordinate.split(' ')
    lat_deg = float(lat[0:2])
    lat_min = float(lat[2:4])
    lat_sec = float(lat[4:6])
    lat_dd = dms_to_dd(lat_deg, lat_min, lat_sec)

    lon_deg = float(lon[0:3])
    lon_min = float(lon[3:5])
    lon_sec = float(lon[5:7])
    lon_dd = dms_to_dd(lon_deg, lon_min, lon_sec)

    if lat[-1] == 'S':
        lat_dd = -lat_dd        
    if lon[-1] == 'W':
        lon_dd = -lon_dd

    return Point(lon_dd, lat_dd)

def convert_coordinate(position_type: str, coordinate: str) -> Point:
    if position_type == "DD_LON, DD_LAT":
        return from_lon_lat(coordinate)
    elif position_type == "MGRS":
        return from_mgrs(coordinate)
    elif position_type == "DMS":
        return from_dms(coordinate)
    else:
        raise TypeError

In [67]:
df['geometry'] = df.apply(
    lambda x: convert_coordinate(x.position_type, x.position),
    axis=1
)
# Convert to gdf
gdf = gpd.GeoDataFrame(df[['name']], geometry=df['geometry'])

In [70]:
m = Map(
    basemap=basemaps.OpenStreetMap.Mapnik,
    center=(59.91228, 10.78560),
    zoom=10
)

geo_data = GeoData(geo_dataframe = gdf,
    name = 'addresser')

m.add(geo_data)
m.add(LayersControl())

m

Map(center=[59.91228, 10.7856], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'z…

NameError: name 'mrgs' is not defined